# Attention Map — 시사/뉴스/사건 서브카테고리
## dim 38 / 606 / 108 × ~34 / 65~ 분리 × 채널당 1장 × Top50

**데이터:** `SOCIETY_new_category_v3.csv` → `시사/뉴스/사건` 필터 + DINOv2 768차원 merge

### 분석 구성
1. Top50 샘플 선택 (dim값 기준, 채널당 1장)
2. Attention Rollout + Dim Activation Map
3. ROI 마스크 (EasyOCR + YOLOv8 + 배경)
4. Deletion / Insertion 정량 분석


## 0. 패키지 확인

In [ ]:
import subprocess, sys
for pkg, name in [("cv2","opencv-python-headless"),("easyocr","easyocr"),("ultralytics","ultralytics")]:
    try: __import__(pkg)
    except ImportError:
        print(f"설치 중: {name}")
        subprocess.check_call([sys.executable,"-m","pip","install","-q",name])
print("패키지 준비 완료")

## 1. 공통 설정

In [ ]:
import os, math, warnings, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image, ImageFilter
import torch
import torch.nn.functional as F
from transformers import AutoImageProcessor, AutoModel
warnings.filterwarnings("ignore")

# ── 한글 폰트 (글자 깨짐 방지) ─────────────────────────────────────────────
from matplotlib import font_manager
for _fp in [
    "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
    "/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf",
]:
    if Path(_fp).exists():
        font_manager.fontManager.addfont(_fp)
matplotlib.rcParams["font.family"] = "NanumGothic"
matplotlib.rcParams["axes.unicode_minus"] = False
print("한글 폰트: NanumGothic")

# ══════════════════════════════════════════════
# ★ 설정 (상대 경로 — 계정/절대경로에 의존하지 않음)
# ══════════════════════════════════════════════
# 노트북은 보통 6_썸네일_attention_gradcam/ 에서 실행
NB_DIR = Path.cwd()
if NB_DIR.name == "6_썸네일_attention_gradcam":
    ROOT = NB_DIR.parent
elif (NB_DIR / "6_썸네일_attention_gradcam").exists():
    ROOT = NB_DIR
else:
    ROOT = NB_DIR.parent

BIN_ROOT = ROOT.parent / "urp_bin" / "SOCIETY 파일들 - 썸네일 사진 분석"

INPUT_CSV       = ROOT / "Data/SOCIETY/SOCIETY_new_category_v3.csv"
DINOV2_CSV      = BIN_ROOT / "SOCIETY_final_kr_clean_with_dinov2_768_fixed.csv"
MERGED_CSV      = Path("data/SOCIETY_news_with_dinov2.csv")  # §2에서 자동 생성
THUMBNAIL_BASE  = ROOT / "Data/SOCIETY/thumbnails"
OUTPUT_DIR      = Path("outputs")
TARGET_SUBCAT   = "시사/뉴스/사건"

TOP_K           = 50
TARGET_DIMS     = [38, 606, 108]
ALPHA           = 0.45
PATH_CANDIDATES = ["resolved_path", "resolved_thumbnail_path", "thumbnail_path", "thumbnail_path_x"]
CHANNEL_COL_CANDIDATES = ["channel_id", "channelId", "channel", "channel_name"]
ROI_CACHE_FILE  = OUTPUT_DIR / "roi_masks_cache_attn_news.pkl"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(ROOT / "6_썸네일_attention_gradcam/data").mkdir(parents=True, exist_ok=True)
CLASS_NAMES = {0: "~34", 1: "65~"}
print(f"device={DEVICE} | subcategory={TARGET_SUBCAT}")
print(f"ROOT={ROOT}")
print(f"DINOV2_CSV exists={DINOV2_CSV.exists()} | {DINOV2_CSV}")
print(f"OUTPUT_DIR={OUTPUT_DIR.resolve()}")


## 2. CSV 로드 (시사/뉴스/사건) + 경로 보정 + merged CSV 저장


In [ ]:
# ── 1) 서브카테고리 필터 + DINOv2 merge ─────────────────────────────────────
df = pd.read_csv(INPUT_CSV)
df = df[df["subcategory"] == TARGET_SUBCAT].copy()
print(f"Subcategory '{TARGET_SUBCAT}': {len(df)} rows")

dino = pd.read_csv(DINOV2_CSV)
dino_cols = [c for c in dino.columns if c.startswith("dinov2_")]
merge_cols = list(dict.fromkeys(["video_id"] + dino_cols))
df = df.merge(dino[merge_cols], on="video_id", how="inner")
df = df.loc[:, ~df.columns.duplicated()]
if "dinov2_image_loaded" in df.columns:
    df = df[df["dinov2_image_loaded"] == True].copy()
print(f"After DINOv2 merge: {len(df)} rows")

MERGED_CSV.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(MERGED_CSV, index=False, encoding="utf-8-sig")
print(f"Saved merged CSV: {MERGED_CSV.resolve()}")

# ── 2) 썸네일 경로 보정 ───────────────────────────────────────────────────────
path_col = next((c for c in PATH_CANDIDATES if c in df.columns), None)
if path_col is None:
    path_col = "thumbnail_path"

def resolve_path(p):
    if not isinstance(p, str):
        return None
    norm = p.replace("\\", "/")
    if os.path.isabs(norm) and os.path.exists(norm):
        return norm
    redundant = "Data/SOCIETY/thumbnails/"
    if redundant in norm:
        norm = norm.split(redundant)[-1]
    candidate = THUMBNAIL_BASE / norm.lstrip("/")
    return str(candidate) if candidate.exists() else None

if path_col != "resolved_path":
    df["resolved_path"] = df[path_col].apply(resolve_path)
    path_col = "resolved_path"
else:
    df["resolved_path"] = df[path_col].apply(resolve_path)

df = df[df["resolved_path"].notna()].copy()

# ── 3) 연령 binary (~34 vs 65~) ─────────────────────────────────────────────
df["~34"]   = df["age_~17"] + df["age_18~24"] + df["age_25~34"]
df["35~44"] = df["age_35~44"]
df["45~54"] = df["age_45~54"]
df["55~64"] = df["age_55~64"]
df["65~"]   = df["age_65~"]
GROUPED_COLS = ["~34", "35~44", "45~54", "55~64", "65~"]
df["y_grouped"] = df[GROUPED_COLS].idxmax(axis=1)
df["target"]    = df["y_grouped"].map({"~34": 0, "65~": 1})
df_bin = df[df["y_grouped"].isin(["~34", "65~"])].copy()

channel_col = next((c for c in CHANNEL_COL_CANDIDATES if c in df_bin.columns), None)
print(f"path_col={path_col} | channel_col={channel_col}")
print(f"binary samples: {len(df_bin)} | ~34: {(df_bin.target==0).sum()} | 65~: {(df_bin.target==1).sum()}")


## 3. Top50 샘플 선택 — 채널당 1장 + ~34/65~ 분리

In [ ]:
def dedup_by_channel(df_sorted, top_n=50):
    """정렬된 df에서 채널당 1장씩, top_n장 선택"""
    if channel_col is None:
        print("  ⚠️  채널 컬럼 없음 → 중복 제거 없이 상위 반환")
        return df_sorted.head(top_n).reset_index(drop=True)
    seen = set()
    rows = []
    for _, row in df_sorted.iterrows():
        ch = row[channel_col]
        if ch not in seen:
            seen.add(ch)
            rows.append(row)
        if len(rows) == top_n:
            break
    result = pd.DataFrame(rows).reset_index(drop=True)
    print(f"  → {len(result)}장 선택 (채널 {result[channel_col].nunique()}개)")
    return result


def get_top50_samples(df_bin, dim_idx, top_k=50):
    """
    dim_idx 기준 상위 top_k장 — ~34 / 65~ 각각
    기존 top/bottom CSV 있으면 재활용
    """
    feat = f"dinov2_{dim_idx:03d}"
    samples = {}

    for age_label, age_target in [("34", 0), ("65", 1)]:
        csv_path = os.path.join(OUTPUT_DIR, f"top{top_k}_dim{dim_idx:03d}_{age_label}.csv")

        # 기존 CSV 재활용
        if os.path.exists(csv_path):
            print(f"[dim {dim_idx} | {age_label}세] 기존 CSV 로드: {csv_path}")
            sub = pd.read_csv(csv_path)
            if "resolved_path" not in sub.columns:
                sub["resolved_path"] = sub[path_col].apply(resolve_path)
            sub = sub[sub["resolved_path"].apply(lambda x: isinstance(x,str) and os.path.exists(x))].copy()
        else:
            print(f"[dim {dim_idx} | {age_label}세] 새로 생성")
            age_df = df_bin[df_bin["target"] == age_target].copy()
            if feat not in age_df.columns:
                raise ValueError(f"{feat} 컬럼 없음")
            sorted_df = age_df.sort_values(feat, ascending=False)
            sub = dedup_by_channel(sorted_df, top_n=top_k)
            sub.to_csv(csv_path, index=False, encoding="utf-8-sig")
            print(f"  저장: {csv_path}")

        samples[age_label] = sub
        print(f"  최종: {len(sub)}장")

    return samples   # {"34": df, "65": df}


# 전체 dim × 연령 샘플 준비
all_samples = {}   # {dim_idx: {"34": df, "65": df}}
for dim_idx in TARGET_DIMS:
    all_samples[dim_idx] = get_top50_samples(df_bin, dim_idx, TOP_K)

## 4. DINOv2 로드 (1회)

In [ ]:
MODEL_NAME = "facebook/dinov2-base"
processor  = AutoImageProcessor.from_pretrained(MODEL_NAME)
dino       = AutoModel.from_pretrained(MODEL_NAME, attn_implementation="eager").to(DEVICE)
dino.config.output_attentions = True
dino.eval()
print("DINOv2 로드 완료")

## 5. Attention / Dim Activation 함수 (GradCAM get_cam 역할)

In [ ]:
# ── GradCAM의 get_pixel_values와 동일한 역할 ─────────────────────────────
def get_pixel_values(img_pil):
    inputs  = processor(images=img_pil, return_tensors="pt")
    pv      = inputs["pixel_values"].to(DEVICE)
    _, _, H, W = pv.shape
    img_resized = img_pil.resize((W, H))
    rgb_np  = np.array(img_resized).astype(np.float32) / 255.0
    return pv, rgb_np, H, W


# ── Attention Rollout (GradCAM의 grayscale_cam 역할) ──────────────────────
@torch.no_grad()
def get_attention_rollout_map(pv):
    """pixel_values → [H_patch, W_patch] attention rollout map (0~1)"""
    outputs = dino(pixel_values=pv, output_attentions=True, return_dict=True)
    rollout = torch.eye(outputs.attentions[0].shape[-1], device=DEVICE).unsqueeze(0)
    for attn in outputs.attentions:
        a = attn.mean(dim=1)
        a = (a + torch.eye(a.size(-1), device=DEVICE).unsqueeze(0))
        a = a / a.sum(dim=-1, keepdim=True)
        rollout = torch.bmm(a, rollout)
    cls_patch = rollout[0, 0, 1:]
    g = int(math.sqrt(cls_patch.shape[0]))
    m = cls_patch.reshape(g, g).cpu().numpy()
    return (m - m.min()) / (m.max() - m.min() + 1e-8)


# ── Dim Activation Map (dim별 patch 활성화) ───────────────────────────────
@torch.no_grad()
def get_dim_activation_map(pv, dim_idx):
    """pixel_values → [H_patch, W_patch] dim activation map (0~1)"""
    outputs = dino(pixel_values=pv, return_dict=True)
    patch   = outputs.last_hidden_state[0, 1:, dim_idx]
    patch   = torch.relu(patch)
    g = int(math.sqrt(patch.shape[0]))
    m = patch.reshape(g, g).cpu().numpy()
    return (m - m.min()) / (m.max() - m.min() + 1e-8)


def resize_map_to(map_2d, H, W):
    t = torch.tensor(map_2d).unsqueeze(0).unsqueeze(0).float()
    return F.interpolate(t, size=(H,W), mode="bilinear", align_corners=False)[0,0].numpy()


print("Attention / Dim 함수 정의 완료")

## 6. ROI 마스크 — GradCAM 코드와 동일 (EasyOCR + YOLOv8)

In [ ]:
import easyocr
from ultralytics import YOLO

print("EasyOCR 초기화 중...")
ocr_reader = easyocr.Reader(["ko","en"], gpu=(DEVICE=="cuda"), verbose=False)
print("YOLOv8 초기화 중...")
yolo_model = YOLO("yolov8n.pt")
print("완료")


def build_roi_masks(image_path, H, W):
    """GradCAM 코드의 build_roi_masks와 동일"""
    img_orig = Image.open(image_path).convert("RGB")
    orig_W, orig_H = img_orig.size
    img_np = np.array(img_orig)

    mask_text   = np.zeros((orig_H, orig_W), dtype=bool)
    mask_person = np.zeros((orig_H, orig_W), dtype=bool)

    # 텍스트 (EasyOCR)
    ocr_results = ocr_reader.readtext(
        img_np, detail=1, paragraph=False, min_size=10,
        text_threshold=0.4, low_text=0.3, link_threshold=0.3,
        width_ths=0.8, contrast_ths=0.05, adjust_contrast=0.7,
    )
    for bbox, text, conf in ocr_results:
        if conf < 0.3: continue
        pts = np.array(bbox, dtype=np.int32)
        x0,y0 = max(0,pts[:,0].min()), max(0,pts[:,1].min())
        x1,y1 = min(orig_W,pts[:,0].max()), min(orig_H,pts[:,1].max())
        mask_text[y0:y1, x0:x1] = True

    # 인물 (YOLOv8)
    for box in yolo_model(img_np, verbose=False)[0].boxes:
        if int(box.cls[0]) != 0 or float(box.conf[0]) < 0.3: continue
        x0,y0,x1,y1 = box.xyxy[0].cpu().numpy().astype(int)
        x0,y0 = max(0,x0), max(0,y0)
        x1,y1 = min(orig_W,x1), min(orig_H,y1)
        mask_person[y0:y1, x0:x1] = True

    mask_bg = ~(mask_text | mask_person)

    def resize_mask(m):
        return np.array(
            Image.fromarray(m.astype(np.uint8)*255).resize((W,H), Image.NEAREST)
        ) > 127

    return {"텍스트": resize_mask(mask_text),
            "인물":   resize_mask(mask_person),
            "배경":   resize_mask(mask_bg)}


def get_roi_masks_cached(image_path, H, W, cache):
    key = (image_path, H, W)
    if key not in cache:
        cache[key] = build_roi_masks(image_path, H, W)
    return cache[key]


def prebuild_roi_cache(sample_dfs, cache_file=ROI_CACHE_FILE):
    if os.path.exists(cache_file):
        print(f"기존 캐시 로드: {cache_file}")
        with open(cache_file, "rb") as f:
            return pickle.load(f)
    cache = {}
    paths = list(dict.fromkeys([p for df in sample_dfs for p in df["resolved_path"].tolist()]))
    print(f"ROI 빌드 시작: {len(paths)}장")
    for i, path in enumerate(paths):
        try:
            cache[(path,224,224)] = build_roi_masks(path,224,224)
            if (i+1)%10==0: print(f"  {i+1}/{len(paths)}")
        except Exception as e:
            print(f"  ⚠️  {os.path.basename(path)}: {e}")
    with open(cache_file,"wb") as f: pickle.dump(cache,f)
    print(f"캐시 저장: {cache_file}")
    return cache


print("ROI 함수 정의 완료")

## 7. ROI 캐시 사전 빌드 (전체 dim × 연령 합쳐서 한 번에)

In [ ]:
all_dfs = []
for dim_idx in TARGET_DIMS:
    for age_label in ["34","65"]:
        all_dfs.append(all_samples[dim_idx][age_label])

roi_cache = prebuild_roi_cache(all_dfs)

## 8. ROI 감지 확인 그리드 (GradCAM show_roi_detection 동일)

In [ ]:
def show_roi_detection(sample_df, roi_cache, n_show=10, save_path="roi_detection_check.png"):
    """열: [원본] [텍스트마스크] [인물마스크] [배경마스크] [ROI합성] — GradCAM 코드와 동일"""
    sample_df = sample_df.head(n_show).reset_index(drop=True)
    ncols = 5
    fig, axes = plt.subplots(n_show, ncols, figsize=(ncols*4, n_show*4))
    if n_show == 1: axes = axes[np.newaxis,:]

    for j,t in enumerate(["원본","텍스트(빨강)","인물(파랑)","배경(초록)","ROI합성"]):
        axes[0,j].set_title(t, fontsize=10, fontweight="bold", pad=8)

    for i,(_,row) in enumerate(sample_df.iterrows()):
        img_orig = Image.open(row["resolved_path"]).convert("RGB")
        pv, rgb_np, H, W = get_pixel_values(img_orig)
        roi = get_roi_masks_cached(row["resolved_path"],H,W,roi_cache)
        tm,pm,bm = roi["텍스트"],roi["인물"],roi["배경"]

        ch    = str(row.get("channel_name",""))
        title = str(row.get("title",""))[:25]

        axes[i,0].imshow((rgb_np*255).astype(np.uint8))
        axes[i,0].set_title(f"{ch}\n{title}", fontsize=6.5); axes[i,0].axis("off")

        for j,(mask,col,lbl) in enumerate([
            (tm,[255,80,80],"텍스트"),
            (pm,[80,120,255],"인물"),
            (bm,[80,200,80],"배경")
        ]):
            vis = (rgb_np*255).astype(np.uint8).copy()
            vis[mask]  = (vis[mask]*0.4 + np.array(col)*0.6).astype(np.uint8)
            vis[~mask] = (vis[~mask]*0.3).astype(np.uint8)
            axes[i,1+j].imshow(vis)
            pct = mask.mean()*100
            axes[i,1+j].set_title(
                ("✅" if mask.any() else "❌")+f" {lbl}: {pct:.1f}%", fontsize=8
            )
            axes[i,1+j].axis("off")

        # ROI 합성
        ov = (rgb_np*255).astype(np.uint8).copy()
        for mask,col in [(tm,[255,80,80]),(pm,[80,120,255]),(bm,[80,200,80])]:
            ov[mask] = (ov[mask]*0.6 + np.array(col)*0.4).astype(np.uint8)
        axes[i,4].imshow(ov)
        axes[i,4].set_title("합성",fontsize=8); axes[i,4].axis("off")

    plt.suptitle(f"ROI 감지 확인 ({n_show}장)  |  🔴텍스트 🔵인물 🟢배경", fontsize=12)
    plt.tight_layout()
    plt.savefig(save_path, dpi=130, bbox_inches="tight")
    print(f"저장: {save_path}")
    plt.close(fig)

## 9. 마스킹 시각화 그리드 (GradCAM masking_visualization_grid 동일 구조)

In [ ]:
def masking_visualization_grid(
    sample_df, roi_cache, dim_idx,
    heat_type="attn",      # "attn" or "dim"
    thresholds=(0.3,0.5,0.7),
    save_path="masking_grid.png"
):
    """
    GradCAM masking_visualization_grid와 동일 구조.
    열: 원본(+ROI 경계) | 히트맵 | thr0.3 | thr0.5 | thr0.7
    heat_type="attn"  → Attention Rollout
    heat_type="dim"   → Dim Activation (dim_idx)
    """
    sample_df = sample_df.reset_index(drop=True)
    n     = len(sample_df)
    ncols = 2 + len(thresholds)
    heat_label = "Attn Rollout" if heat_type=="attn" else f"Dim {dim_idx} Act"

    fig, axes = plt.subplots(n, ncols, figsize=(4.5*ncols, 4*n))
    if n == 1: axes = axes[np.newaxis,:]

    ROI_COLORS = {"텍스트":[255,80,80], "인물":[80,120,255], "배경":[80,200,80]}

    for i,(_,row) in enumerate(sample_df.iterrows()):
        img_orig = Image.open(row["resolved_path"]).convert("RGB")
        pv, rgb_np, H, W = get_pixel_values(img_orig)

        # 히트맵 선택
        if heat_type == "attn":
            raw_map = get_attention_rollout_map(pv)
        else:
            raw_map = get_dim_activation_map(pv, dim_idx)
        heat_full = resize_map_to(raw_map, H, W)

        roi_masks = get_roi_masks_cached(row["resolved_path"],H,W,roi_cache)

        # 원본 + ROI 반투명 오버레이
        overlay = (rgb_np*255).astype(np.uint8).copy()
        for rname,rmask in roi_masks.items():
            c = np.array(ROI_COLORS[rname],dtype=np.uint8)
            overlay[rmask] = (overlay[rmask]*0.6 + c*0.4).astype(np.uint8)

        ch    = str(row.get("channel_name",""))
        title = str(row.get("title",""))[:28]
        feat  = f"dinov2_{dim_idx:03d}"
        fval  = row.get(feat, float("nan"))
        age   = row.get("y_grouped","?")

        axes[i,0].imshow(overlay)
        axes[i,0].set_title(
            f"{age} | {feat}={fval:.3f}\n{ch} | {title}\n🔴텍스트 🔵인물 🟢배경",
            fontsize=6.5
        )
        axes[i,0].axis("off")

        axes[i,1].imshow(heat_full, cmap="jet", vmin=0, vmax=1)
        axes[i,1].set_title(heat_label, fontsize=8)
        axes[i,1].axis("off")

        for j,thr in enumerate(thresholds):
            mask   = heat_full >= thr
            masked = rgb_np.copy()
            masked[~mask] = 0.0
            axes[i,2+j].imshow((masked*255).astype(np.uint8))
            axes[i,2+j].set_title(f"thr≥{thr:.1f} | {mask.mean()*100:.0f}%", fontsize=8)
            axes[i,2+j].axis("off")

    plt.suptitle(
        f"마스킹 그리드 [{heat_label}] dim={dim_idx} ({n}장)  |  🔴텍스트 🔵인물 🟢배경",
        fontsize=12
    )
    plt.tight_layout()
    plt.savefig(save_path, dpi=130, bbox_inches="tight")
    print(f"저장: {save_path}")
    plt.close(fig)


print("masking_visualization_grid 정의 완료")

## 10. ROI 비중 정량 분석 함수 (Deletion / Insertion)

In [ ]:
# ── ROI 영역 비중 수치화 ─────────────────────────────────────────────────────
def compute_region_scores(heat_map, roi_masks):
    """각 ROI에서의 attention 비중 계산"""
    total = heat_map.sum() + 1e-8
    return {k: float((heat_map * m.astype(np.float32)).sum() / total)
            for k, m in roi_masks.items()}


# ── Deletion / Insertion (GradCAM roi_deletion_insertion 동일 구조) ──────────
def roi_deletion_insertion_attn(image_path, roi_cache, dim_idx, heat_type="attn"):
    """
    heat_type: "attn" (Attention Rollout) or "dim" (Dim Activation)
    각 ROI를 지웠을 때(Deletion) / 남겼을 때(Insertion) 의 히트맵 변화 측정
    → attention은 확률이 아니므로, mean activation값을 사용
    """
    img_orig = Image.open(image_path).convert("RGB")
    pv, rgb_np, H, W = get_pixel_values(img_orig)
    blurred = np.array(
        img_orig.resize((W,H)).filter(ImageFilter.GaussianBlur(radius=11))
    ).astype(np.float32)/255.0
    mean_val = rgb_np.mean(axis=(0,1), keepdims=True)

    # 원본 히트맵 (mean activation)
    if heat_type=="attn":
        orig_map = resize_map_to(get_attention_rollout_map(pv), H, W)
    else:
        orig_map = resize_map_to(get_dim_activation_map(pv, dim_idx), H, W)
    orig_mean = float(orig_map.mean())

    roi_masks = get_roi_masks_cached(image_path,H,W,roi_cache)
    if not roi_masks["텍스트"].any() and not roi_masks["인물"].any():
        roi_masks["배경"] = np.ones((H,W), dtype=bool)

    results = {}
    for roi_name, mask in roi_masks.items():
        m = mask[...,None].astype(np.float32)

        # Deletion: 해당 ROI를 평균값으로 채움
        del_img = Image.fromarray(((rgb_np*(1-m) + mean_val*m)*255).astype(np.uint8))
        del_pv, _, _, _ = get_pixel_values(del_img)
        if heat_type=="attn":
            del_map = resize_map_to(get_attention_rollout_map(del_pv),H,W)
        else:
            del_map = resize_map_to(get_dim_activation_map(del_pv,dim_idx),H,W)
        del_mean = float(del_map.mean())

        # Insertion: 해당 ROI만 남기고 나머지는 블러
        ins_img = Image.fromarray(((rgb_np*m + blurred*(1-m))*255).astype(np.uint8))
        ins_pv, _, _, _ = get_pixel_values(ins_img)
        if heat_type=="attn":
            ins_map = resize_map_to(get_attention_rollout_map(ins_pv),H,W)
        else:
            ins_map = resize_map_to(get_dim_activation_map(ins_pv,dim_idx),H,W)
        ins_mean = float(ins_map.mean())

        # 비중
        region_score = float((orig_map * mask.astype(np.float32)).sum() / (orig_map.sum()+1e-8))

        results[roi_name] = {
            "score":  region_score,   # 원본에서의 attention 비중
            "del":    del_mean,        # 지웠을 때 mean activation
            "ins":    ins_mean,        # 남겼을 때 mean activation
        }

    return results, orig_mean


# ── 배치: ROI 막대그래프 (GradCAM batch_roi_analysis 동일) ─────────────────
def batch_roi_analysis(
    sample_df, label, roi_cache, dim_idx,
    heat_type="attn", n_samples=50,
    save_path="roi_analysis.png"
):
    sample_df = sample_df.head(n_samples).reset_index(drop=True)
    roi_scores, roi_del, roi_ins, orig_list = {},{},{}, []

    for _,row in sample_df.iterrows():
        try:
            res, orig_m = roi_deletion_insertion_attn(
                row["resolved_path"], roi_cache, dim_idx, heat_type
            )
            orig_list.append(orig_m)
            for rname,vals in res.items():
                roi_scores.setdefault(rname,[]).append(vals["score"])
                roi_del.setdefault(rname,[]).append(vals["del"])
                roi_ins.setdefault(rname,[]).append(vals["ins"])
        except Exception as e:
            print(f"  ⚠️  건너뜀: {e}")

    if not roi_del: print("유효 샘플 없음"); return {}

    heat_label = "Attn Rollout" if heat_type=="attn" else f"Dim {dim_idx}"
    roi_names  = list(roi_del.keys())
    mean_orig  = np.mean(orig_list)
    mean_scores = [np.mean(roi_scores[r]) for r in roi_names]
    mean_dels   = [np.mean(roi_del[r])    for r in roi_names]
    mean_ins_v  = [np.mean(roi_ins[r])    for r in roi_names]

    x      = np.arange(len(roi_names))
    colors = ["#e74c3c","#3498db","#2ecc71"]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # ① attention 비중 막대
    bars = axes[0].bar(x, mean_scores, 0.4, color=colors[:len(roi_names)], alpha=0.85)
    axes[0].set_title(f"ROI별 Attention 비중\n{heat_label} — {label}", fontsize=11)
    axes[0].set_xticks(x); axes[0].set_xticklabels(roi_names)
    axes[0].set_ylabel("attention 비중"); axes[0].set_ylim(0,1)
    axes[0].grid(axis="y",alpha=0.3)
    for bar,val in zip(bars,mean_scores):
        axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                    f"{val:.3f}",ha="center",va="bottom",fontsize=11,fontweight="bold")

    # ② Deletion 막대
    for ax,vals,direction,ylabel in [
        (axes[1], mean_dels,  "Deletion  (지웠을 때)",  "Mean Activation (ROI 제거 후)"),
        (axes[2], mean_ins_v, "Insertion (남겼을 때)",  "Mean Activation (ROI만 공개)"),
    ]:
        bars = ax.bar(x, vals, 0.4, color=colors[:len(roi_names)], alpha=0.85)
        ax.axhline(mean_orig, color="black", linestyle="--", linewidth=1.5,
                   label=f"원본 ({mean_orig:.3f})")
        ax.set_title(f"ROI {direction}\n{heat_label} — {label}", fontsize=11)
        ax.set_xticks(x); ax.set_xticklabels(roi_names)
        ax.set_ylabel(ylabel); ax.set_ylim(0, mean_orig*2+0.01)
        ax.legend(); ax.grid(axis="y",alpha=0.3)
        for bar,val in zip(bars,vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+mean_orig*0.02,
                   f"{val:.3f}",ha="center",va="bottom",fontsize=11,fontweight="bold")

    plt.suptitle(
        f"ROI별 정량 분석  [{label}]  heat={heat_label}  dim={dim_idx}\n"
        f"원본 평균 activation: {mean_orig:.4f}  |  n={len(orig_list)}개",
        fontsize=12
    )
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    print(f"저장: {save_path}")
    plt.close(fig)

    print(f"\n  [{label}|{heat_label}] 요약  원본:{mean_orig:.4f}")
    print(f"  {'ROI':<6}  비중    Deletion  Insertion")
    for rn,sc,d,ins in zip(roi_names,mean_scores,mean_dels,mean_ins_v):
        print(f"  {rn:<6}  {sc:.3f}   {d:.4f}    {ins:.4f}")

    return {"orig": mean_orig, "scores": dict(zip(roi_names,mean_scores)),
            "del": dict(zip(roi_names,mean_dels)), "ins": dict(zip(roi_names,mean_ins_v))}


print("정량 분석 함수 정의 완료")

## 11. 메인 실행 함수

In [ ]:
def run_full_analysis(dim_idx, n_samples=TOP_K):
    """
    dim_idx에 대해 ~34 / 65~ 각각:
      1. ROI 감지 확인 그리드
      2. 마스킹 시각화 그리드 (Attn / Dim 각각, 25장×2파일)
      3. ROI 정량 분석 막대그래프 (Attn / Dim 각각)
    """
    feat = f"dinov2_{dim_idx:03d}"
    summary_records = {}

    for age_label, age_str in [("34","~34"),("65","65~")]:
        sub = all_samples[dim_idx][age_label]
        print(f"\n{'='*60}")
        print(f" dim {dim_idx} | {age_str} | {len(sub)}장")
        print(f"{'='*60}")

        base = os.path.join(OUTPUT_DIR, f"dim{dim_idx:03d}_{age_label}")
        os.makedirs(base, exist_ok=True)

        # ── (1) ROI 감지 확인 (처음 10장)
        print("[1] ROI 감지 확인 그리드")
        show_roi_detection(
            sub, roi_cache, n_show=10,
            save_path=f"{base}/roi_detection.png"
        )

        for heat_type, heat_name in [("attn","attn"),("dim","dim")]:
            print(f"\n[2+3] heat_type={heat_name}")

            # ── (2) 마스킹 시각화 그리드 (25장 × 2파일)
            print("  마스킹 그리드 (1/2)")
            masking_visualization_grid(
                sub.iloc[:25], roi_cache, dim_idx,
                heat_type=heat_type,
                save_path=f"{base}/masking_{heat_name}_grid1.png"
            )
            if len(sub) > 25:
                print("  마스킹 그리드 (2/2)")
                masking_visualization_grid(
                    sub.iloc[25:], roi_cache, dim_idx,
                    heat_type=heat_type,
                    save_path=f"{base}/masking_{heat_name}_grid2.png"
                )

            # ── (3) ROI 정량 분석
            print("  ROI 정량 분석")
            rec = batch_roi_analysis(
                sub, label=f"{age_str} | top{n_samples}",
                roi_cache=roi_cache, dim_idx=dim_idx,
                heat_type=heat_type, n_samples=n_samples,
                save_path=f"{base}/roi_analysis_{heat_name}.png"
            )
            summary_records[f"{age_label}_{heat_name}"] = rec

    print(f"\n✅ dim {dim_idx} 완료")
    return summary_records


print("run_full_analysis 정의 완료")

## 12. ★ dim 38 실행

In [ ]:
summary_38 = run_full_analysis(dim_idx=38)

## 13. ★ dim 606 실행

In [ ]:
summary_606 = run_full_analysis(dim_idx=606)

## 14. ★ dim 108 실행

In [ ]:
summary_108 = run_full_analysis(dim_idx=108)

## 15. 통합 비교 — dim 3개 × 연령 2그룹 히트맵

In [ ]:
all_summaries = {38: summary_38, 606: summary_606, 108: summary_108}

for heat_type, heat_name in [("attn","Attention Rollout"),("dim","Dim Activation")]:
    fig, axes = plt.subplots(1, 2, figsize=(18, 5))

    for ax, age_label, age_str in [(axes[0],"34","~34"),(axes[1],"65","65~")]:
        key  = f"{age_label}_{heat_type}"
        rows, row_labels = [], []

        for dim_idx in TARGET_DIMS:
            rec = all_summaries[dim_idx].get(key, {})
            scores = rec.get("scores", {"텍스트":0,"인물":0,"배경":0})
            rows.append([scores.get("텍스트",0), scores.get("인물",0), scores.get("배경",0)])
            row_labels.append(f"dim {dim_idx}")

        mat = np.array(rows)
        im  = ax.imshow(mat, cmap="RdYlGn", vmin=0, vmax=0.8, aspect="auto")
        ax.set_xticks([0,1,2]); ax.set_xticklabels(["텍스트","인물","배경"],fontsize=12)
        ax.set_yticks(range(len(row_labels))); ax.set_yticklabels(row_labels,fontsize=12)
        ax.set_title(f"{age_str} — attention 비중",fontsize=13)
        plt.colorbar(im,ax=ax,fraction=0.046)
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                ax.text(j,i,f"{mat[i,j]:.3f}",ha="center",va="center",fontsize=11,
                        color="black" if mat[i,j]<0.55 else "white")

    plt.suptitle(f"dim 38/606/108 × ~34/65~ — ROI 비중 [{heat_name}]",fontsize=14)
    plt.tight_layout()
    sp = os.path.join(OUTPUT_DIR, f"summary_heatmap_{heat_type}.png")
    plt.savefig(sp, dpi=150, bbox_inches="tight")
    print(f"저장: {sp}")
    plt.show()

## 16. (Optional) 특정 이미지 단독 빠른 확인

In [ ]:
# quick_image_path / quick_dim 바꿔서 실행 (로컬 썸네일 경로로 수정)
quick_image_path = str(THUMBNAIL_BASE / "0시기록" / "HssVM_6vg9c.jpg")
quick_dim        = 38   # 38 / 606 / 108

if os.path.exists(quick_image_path):
    img = Image.open(quick_image_path).convert("RGB")
    pv, rgb_np, H, W = get_pixel_values(img)
    attn_map = resize_map_to(get_attention_rollout_map(pv), H, W)
    dim_map  = resize_map_to(get_dim_activation_map(pv, quick_dim), H, W)
    roi      = build_roi_masks(quick_image_path, H, W)

    fig, axes = plt.subplots(2, 4, figsize=(22, 10))
    axes[0,0].imshow((rgb_np*255).astype(np.uint8)); axes[0,0].set_title("Original"); axes[0,0].axis("off")
    axes[0,1].imshow(attn_map, cmap="jet"); axes[0,1].set_title("Attn Rollout"); axes[0,1].axis("off")
    for j,(rname,rmask) in enumerate(roi.items()):
        vis = (rgb_np*255).astype(np.uint8).copy()
        cols = [[255,80,80],[80,120,255],[80,200,80]]
        vis[rmask]  = (vis[rmask]*0.4 + np.array(cols[j])*0.6).astype(np.uint8)
        vis[~rmask] = (vis[~rmask]*0.3).astype(np.uint8)
        axes[0,2+j].imshow(vis)
        axes[0,2+j].set_title(f"ROI: {rname}  ({rmask.mean()*100:.0f}%)"); axes[0,2+j].axis("off")

    axes[1,0].imshow(dim_map, cmap="jet"); axes[1,0].set_title(f"Dim {quick_dim} Act"); axes[1,0].axis("off")
    for j,(rname,rmask) in enumerate(roi.items()):
        for col_idx, (heat, heat_nm) in enumerate([(attn_map,"Attn"),(dim_map,f"Dim{quick_dim}")]):
            sc = float((heat*rmask.astype(np.float32)).sum()/(heat.sum()+1e-8))
            # 마스크 적용 히트맵
            mh = heat*rmask.astype(np.float32)
    # 마스킹 히트맵 row
    for j,(rname,rmask) in enumerate(roi.items()):
        mh = attn_map * rmask.astype(np.float32)
        sc = float((attn_map*rmask.astype(np.float32)).sum()/(attn_map.sum()+1e-8))
        axes[1,1+j].imshow(mh, cmap="jet"); axes[1,1+j].axis("off")
        axes[1,1+j].set_title(f"Attn×{rname}\n비중={sc:.3f}")

    plt.suptitle(f"단독 확인: dim={quick_dim} | {os.path.basename(quick_image_path)}", fontsize=12)
    plt.tight_layout()
    sp = os.path.join(OUTPUT_DIR, f"quick_dim{quick_dim}.png")
    plt.savefig(sp, dpi=130, bbox_inches="tight")
    print(f"저장: {sp}")
    plt.close()
else:
    print("경로 없음. quick_image_path 수정 필요")